# IMS Regrid QC Over Colorado (Raw vs M36-Regridded)

This notebook makes side-by-side map checks for selected days to verify that
`regrid_ims_to_m36_nearest.py` is behaving as expected.

For each date, the figure is a 1x2 panel:
- Left: original IMS 24 km category grid
- Right: IMS category regridded to M36 EASE


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

YEAR = 2020

# Adjust these paths if needed.
IMS_RAW_FILE = Path("/gpfsm/dnb06/projects/p163/IMS/ims_snowcover_24km_2020.nc4")
IMS_REGRID_FILE = Path("/discover/nobackup/projects/land_da/geosldas-analysis/projects/IMS/output/ims_snowcover_24km_2020_on_m36_nearest.nc4")

RAW_VAR_CANDIDATES = ("ims_snowcover", "ims_category")
REGRID_VAR_CANDIDATES = ("ims_category", "ims_snowcover")

DATES_TO_PLOT = [
    pd.Timestamp("2020-01-01"),
    pd.Timestamp("2020-04-01"),
    pd.Timestamp("2020-11-01"),
]

# Colorado map extent: (west, east, south, north).
CO_EXTENT = (-109.1, -102.0, 36.8, 41.2)

IMS_FILL_VALUES = {-32768}

print(f"IMS_RAW_FILE={IMS_RAW_FILE}")
print(f"IMS_REGRID_FILE={IMS_REGRID_FILE}")
print(f"DATES_TO_PLOT={[d.strftime('%Y-%m-%d') for d in DATES_TO_PLOT]}")


In [ ]:
def choose_var(ds: xr.Dataset, candidates):
    for name in candidates:
        if name in ds.variables:
            return name
    raise KeyError(f"None of the candidate variables were found: {candidates}")


def wrap_lon_180(lon):
    arr = np.asarray(lon, dtype=np.float64)
    return ((arr + 180.0) % 360.0) - 180.0


def decode_ims_dates(ds: xr.Dataset, var_name: str, year: int) -> pd.DatetimeIndex:
    da = ds[var_name]
    if da.ndim != 3:
        raise ValueError(f"{var_name} must be 3D; got dims={da.dims}")

    n_time = int(da.shape[0])
    time_dim = str(da.dims[0])

    if "doy" in ds.variables and ds["doy"].ndim == 1 and ds["doy"].shape[0] == n_time:
        doy = np.asarray(ds["doy"].values, dtype=float)
        base = pd.Timestamp(f"{year}-01-01")
        return pd.DatetimeIndex(base + pd.to_timedelta(doy - 1.0, unit="D"))

    for name in (time_dim, "time", "day_of_year"):
        if name not in ds.variables:
            continue
        tvar = ds[name]
        if tvar.ndim != 1 or tvar.shape[0] != n_time:
            continue

        vals = np.asarray(tvar.values)
        if np.issubdtype(vals.dtype, np.datetime64):
            return pd.DatetimeIndex(pd.to_datetime(vals))

        units = str(tvar.attrs.get("units", ""))
        if "since" in units:
            try:
                base_txt = units.split("since", 1)[1].strip().split()[0]
                base = pd.Timestamp(base_txt)
                return pd.DatetimeIndex(base + pd.to_timedelta(vals.astype(float), unit="D"))
            except Exception:
                pass

    base = pd.Timestamp(f"{year}-01-01")
    return pd.DatetimeIndex(base + pd.to_timedelta(np.arange(n_time), unit="D"))


def get_date_index(dates: pd.DatetimeIndex, target_day: pd.Timestamp) -> int:
    target = pd.Timestamp(target_day).normalize()
    idx = np.where(dates.normalize() == target)[0]
    if idx.size == 0:
        raise KeyError(f"Date {target.strftime('%Y-%m-%d')} not found in dataset")
    return int(idx[0])


def read_slice_as_yx(da: xr.DataArray, t_index: int, lat2d: np.ndarray):
    time_dim = str(da.dims[0])
    arr = np.asarray(da.isel({time_dim: t_index}).values, dtype=np.float32)

    if arr.shape == lat2d.shape:
        return arr
    if arr.T.shape == lat2d.shape:
        return arr.T

    raise ValueError(
        f"Slice shape {arr.shape} does not match lat/lon shape {lat2d.shape} (or transpose {arr.T.shape})"
    )


def prepare_ims_codes(arr2d: np.ndarray, fill_values: set[int]):
    arr = np.asarray(arr2d, dtype=np.float32)
    finite = np.isfinite(arr)

    out = np.full(arr.shape, np.nan, dtype=np.float32)
    out[finite] = np.rint(arr[finite]).astype(np.float32)

    for fv in fill_values:
        out[out == float(fv)] = np.nan

    return out


# Categorical palette for IMS classes 0..4.
IMS_CODES = [0, 1, 2, 3, 4]
IMS_LABELS = [
    "0: outside coverage",
    "1: water",
    "2: land no snow",
    "3: sea ice",
    "4: snow",
]
IMS_COLORS = ["#cfcfcf", "#2b83ba", "#c7a76c", "#98d5ef", "#ffffff"]
IMS_CMAP = ListedColormap(IMS_COLORS)
IMS_NORM = BoundaryNorm(np.array([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]), IMS_CMAP.N)


In [ ]:
if not IMS_RAW_FILE.exists():
    raise FileNotFoundError(f"Raw IMS file not found: {IMS_RAW_FILE}")
if not IMS_REGRID_FILE.exists():
    raise FileNotFoundError(f"Regridded IMS file not found: {IMS_REGRID_FILE}")

ds_raw = xr.open_dataset(IMS_RAW_FILE, decode_times=False)
ds_regrid = xr.open_dataset(IMS_REGRID_FILE, decode_times=False)

raw_var = choose_var(ds_raw, RAW_VAR_CANDIDATES)
regrid_var = choose_var(ds_regrid, REGRID_VAR_CANDIDATES)

raw_dates = decode_ims_dates(ds_raw, raw_var, YEAR)
regrid_dates = decode_ims_dates(ds_regrid, regrid_var, YEAR)

raw_lat = np.asarray(ds_raw["lat"].values, dtype=np.float32)
raw_lon = wrap_lon_180(np.asarray(ds_raw["lon"].values, dtype=np.float32))

re_lat = np.asarray(ds_regrid["lat"].values, dtype=np.float32)
re_lon = wrap_lon_180(np.asarray(ds_regrid["lon"].values, dtype=np.float32))

print(f"Raw var={raw_var}, shape={ds_raw[raw_var].shape}")
print(f"Regridded var={regrid_var}, shape={ds_regrid[regrid_var].shape}")
print(f"Raw date span: {raw_dates.min()} to {raw_dates.max()}")
print(f"Regrid date span: {regrid_dates.min()} to {regrid_dates.max()}")


In [ ]:
for day in DATES_TO_PLOT:
    i_raw = get_date_index(raw_dates, day)
    i_re = get_date_index(regrid_dates, day)

    raw_slice = read_slice_as_yx(ds_raw[raw_var], i_raw, raw_lat)
    re_slice = read_slice_as_yx(ds_regrid[regrid_var], i_re, re_lat)

    raw_plot = prepare_ims_codes(raw_slice, IMS_FILL_VALUES)
    re_plot = prepare_ims_codes(re_slice, IMS_FILL_VALUES)

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 5),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True,
    )

    pan = [
        (axes[0], raw_lon, raw_lat, raw_plot, "IMS 24km raw"),
        (axes[1], re_lon, re_lat, re_plot, "IMS regridded to M36 (nearest)"),
    ]

    mappable = None
    for k, (ax, lon2d, lat2d, arr2d, title) in enumerate(pan):
        mappable = ax.pcolormesh(
            lon2d,
            lat2d,
            arr2d,
            cmap=IMS_CMAP,
            norm=IMS_NORM,
            transform=ccrs.PlateCarree(),
            shading="auto",
        )

        ax.set_extent(CO_EXTENT, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.7, edgecolor="black")
        ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.5, edgecolor="black")
        ax.coastlines("50m", linewidth=0.4)

        gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False
        if k == 1:
            gl.left_labels = False

        ax.set_title(title)

    cbar = fig.colorbar(mappable, ax=axes, shrink=0.9, pad=0.02)
    cbar.set_ticks(IMS_CODES)
    cbar.set_ticklabels(IMS_LABELS)

    fig.suptitle(f"Colorado IMS QC: {day.strftime('%Y-%m-%d')}", fontsize=13)
    plt.show()


In [ ]:
# Optional quick numeric check: category counts inside Colorado extent for each panel/day.

def code_counts_in_extent(data2d, lon2d, lat2d, extent, valid_codes=(0, 1, 2, 3, 4)):
    w, e, s, n = extent
    m = (lon2d >= w) & (lon2d <= e) & (lat2d >= s) & (lat2d <= n) & np.isfinite(data2d)
    if not np.any(m):
        return {int(c): 0 for c in valid_codes}
    vals = np.rint(data2d[m]).astype(np.int32)
    out = {int(c): int(np.sum(vals == int(c))) for c in valid_codes}
    return out

for day in DATES_TO_PLOT:
    i_raw = get_date_index(raw_dates, day)
    i_re = get_date_index(regrid_dates, day)

    raw_slice = prepare_ims_codes(read_slice_as_yx(ds_raw[raw_var], i_raw, raw_lat), IMS_FILL_VALUES)
    re_slice = prepare_ims_codes(read_slice_as_yx(ds_regrid[regrid_var], i_re, re_lat), IMS_FILL_VALUES)

    c_raw = code_counts_in_extent(raw_slice, raw_lon, raw_lat, CO_EXTENT)
    c_re = code_counts_in_extent(re_slice, re_lon, re_lat, CO_EXTENT)

    print(day.strftime('%Y-%m-%d'))
    print('  raw    :', c_raw)
    print('  regrid :', c_re)

# Close datasets when done.
# ds_raw.close(); ds_regrid.close()
